# ITSentinelNet — version datasets réels Hugging Face (IT, taille moyenne)

Même réseau **custom from scratch** (tokenizer BPE + Transformer encodeur codé à la main,
aucun fine-tuning) que la version précédente, mais entraîné sur de **vrais datasets IT
publiés sur le Hugging Face Hub**, plutôt que sur des données synthétiques.

## Catalogue de 10 datasets HF IT de taille moyenne (~1K à ~1M lignes)

| # | Dataset HF | Domaine IT | Taille approx. |
|---|------------|------------|-----------------|
| 1 | `Tobi-Bueck/customer-support-tickets` | Tickets support IT/technique (EN/DE) | 61.8k lignes |
| 2 | `siddhantdotexe123/help-desk-tickets` | ITSM (tickets, agents, SLA) | ~34k lignes (échantillon 3k) |
| 3 | `interneuronai/it_support_ticket_classification_pegasus_dataset` | Classification tickets IT | 1K-10K lignes |
| 4 | `interneuronai/customer_support_ticket_classification_pegasus_dataset` | Classification tickets support | 1K-10K lignes |
| 5 | `w1z4rd3k/it-support-l1-ticket-classification` | Tickets support L1 (EN/CS) | < 1K lignes |
| 6 | `synthlab0/network-anomaly-logs-fr` | Logs réseau / cybersécurité (FR/EN) | 10K-100K lignes |
| 7 | `honicky/hdfs-logs-encoded-blocks` | Logs système HDFS, détection d'anomalie | 100K-1M séquences |
| 8 | `AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.0` | Cybersécurité défensive (SOC, DevSecOps, cloud) | 83.9k lignes |
| 9 | `DetectVul/devign` | Détection de vulnérabilités dans du code Python | 21.5k fonctions |
| 10 | `vansh11/Network_Anomaly_Detection` | Détection d'anomalies réseau | 100K-1M lignes |

Le dataset **#1** sert de dataset principal d'entraînement (il a déjà des colonnes
`queue` = catégorie et `priority` = priorité, exactement ce dont notre modèle multi-tâches
a besoin). Les 9 autres sont fournis avec du code de chargement/inspection prêt à l'emploi
pour étendre le projet (sécurité réseau, logs, code vulnérable...).

> ⚠️ Le chargement de datasets Hugging Face nécessite un accès internet complet au Hub
> (`huggingface.co`) — ce qui est disponible sur Google Colab par défaut. Exécutez ce
> notebook directement dans Colab (pas dans un environnement sandboxé sans accès réseau).


In [7]:
!pip install -q datasets

import torch
print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)


PyTorch version: 2.11.0+cu128
Device utilisé : cuda


## 1. Catalogue des 10 datasets IT (Hugging Face Hub)

In [8]:
DATASET_CATALOG = {
    "tickets_main": {
        "hf_id": "Tobi-Bueck/customer-support-tickets",
        "domain": "Tickets support IT/technique (EN/DE)",
        "note": "Colonnes: subject, body, answer, type, queue, priority, language, tag_1..tag_8",
    },
    "helpdesk_itsm": {
        "hf_id": "siddhantdotexe123/help-desk-tickets",
        "domain": "ITSM multi-tables (tickets, agents, SLA)",
        "note": "Dataset multi-tables (config 'tickets', 'agents', 'categories', 'comments')",
    },
    "it_support_pegasus": {
        "hf_id": "interneuronai/it_support_ticket_classification_pegasus_dataset",
        "domain": "Classification de tickets IT",
        "note": "CSV unique, ~1K-10K lignes",
    },
    "customer_support_pegasus": {
        "hf_id": "interneuronai/customer_support_ticket_classification_pegasus_dataset",
        "domain": "Classification de tickets support",
        "note": "CSV unique, ~1K-10K lignes",
    },
    "it_support_l1": {
        "hf_id": "w1z4rd3k/it-support-l1-ticket-classification",
        "domain": "Tickets support L1 (EN/CS)",
        "note": "Petit dataset JSON (< 1K lignes), QA + classification",
    },
    "network_anomaly_fr": {
        "hf_id": "synthlab0/network-anomaly-logs-fr",
        "domain": "Logs réseau / cybersécurité (FR/EN)",
        "note": "Tags: cybersecurity, anomaly-detection, network-logs, intrusion-detection",
    },
    "hdfs_logs": {
        "hf_id": "honicky/hdfs-logs-encoded-blocks",
        "domain": "Logs système HDFS, détection d'anomalie",
        "note": "Colonnes: block_id, event_encoded, tokenized_block, label (Normal/Anomaly)",
    },
    "cybersec_fenrir": {
        "hf_id": "AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.0",
        "domain": "Cybersécurité défensive (SOC, DevSecOps, cloud, IR)",
        "note": "Format instruction (system/user/assistant), 83.9k lignes",
    },
    "code_vuln_devign": {
        "hf_id": "DetectVul/devign",
        "domain": "Détection de vulnérabilités dans du code Python",
        "note": "21.5k fonctions, labellisées vulnérable / non-vulnérable",
    },
    "network_anomaly": {
        "hf_id": "vansh11/Network_Anomaly_Detection",
        "domain": "Détection d'anomalies réseau",
        "note": "Format texte brut, 100K-1M lignes",
    },
}

for key, info in DATASET_CATALOG.items():
    print(f"[{key}] {info['hf_id']}")
    print(f"    Domaine : {info['domain']}")
    print(f"    Note    : {info['note']}\n")


[tickets_main] Tobi-Bueck/customer-support-tickets
    Domaine : Tickets support IT/technique (EN/DE)
    Note    : Colonnes: subject, body, answer, type, queue, priority, language, tag_1..tag_8

[helpdesk_itsm] siddhantdotexe123/help-desk-tickets
    Domaine : ITSM multi-tables (tickets, agents, SLA)
    Note    : Dataset multi-tables (config 'tickets', 'agents', 'categories', 'comments')

[it_support_pegasus] interneuronai/it_support_ticket_classification_pegasus_dataset
    Domaine : Classification de tickets IT
    Note    : CSV unique, ~1K-10K lignes

[customer_support_pegasus] interneuronai/customer_support_ticket_classification_pegasus_dataset
    Domaine : Classification de tickets support
    Note    : CSV unique, ~1K-10K lignes

[it_support_l1] w1z4rd3k/it-support-l1-ticket-classification
    Domaine : Tickets support L1 (EN/CS)
    Note    : Petit dataset JSON (< 1K lignes), QA + classification

[network_anomaly_fr] synthlab0/network-anomaly-logs-fr
    Domaine : Logs réseau

## 2. Inspecter n'importe quel dataset du catalogue

Utile pour découvrir les colonnes exactes avant de l'intégrer au pipeline
(chaque dataset a un schéma différent).

In [9]:
from datasets import load_dataset

def inspect_dataset(key, split="train", n=2):
    """Charge un dataset du catalogue et affiche ses colonnes + quelques exemples."""
    hf_id = DATASET_CATALOG[key]["hf_id"]
    print(f"Chargement de {hf_id} ...")
    ds = load_dataset(hf_id, split=split)
    print("Colonnes :", ds.column_names)
    print(f"Nombre de lignes : {len(ds)}")
    for i in range(min(n, len(ds))):
        print("---", ds[i])
    return ds

# Exemple : inspecter le dataset de logs réseau
# ds_preview = inspect_dataset("network_anomaly_fr")


## 3. Chargement et préparation du dataset principal (`Tobi-Bueck/customer-support-tickets`)

On ne garde que les files ("queue") réellement IT, on fusionne `subject` + `body`
en un seul texte, et on harmonise la priorité (`low/medium/high` -> `Low/Medium/Critical`).

In [10]:
raw_ds = load_dataset("Tobi-Bueck/customer-support-tickets", split="train")
print("Colonnes originales :", raw_ds.column_names)
print("Nombre total de lignes :", len(raw_ds))

# On garde uniquement les files IT (on exclut Billing, HR, Sales, Returns, General Inquiry...)
IT_QUEUES = {"Technical Support", "Product Support", "IT Support", "Service Outages and Maintenance"}

def is_it_ticket(example):
    return example["queue"] in IT_QUEUES and example["subject"] is not None and example["body"] is not None

it_ds = raw_ds.filter(is_it_ticket)
print("Nombre de tickets IT retenus :", len(it_ds))

CATEGORIES = sorted(IT_QUEUES)
PRIORITIES = ["Low", "Medium", "Critical"]
CATEGORY2ID = {c: i for i, c in enumerate(CATEGORIES)}
PRIORITY2ID = {p: i for i, p in enumerate(PRIORITIES)}

# Le dataset original utilise low/medium/high -> on harmonise vers notre échelle
RAW_PRIORITY_MAP = {"low": "Low", "medium": "Medium", "high": "Critical"}

def normalize_priority(value):
    return RAW_PRIORITY_MAP.get(str(value).strip().lower(), "Medium")

print("Catégories retenues :", CATEGORIES)


README.md:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

aa_dataset-tickets-multi-lang-5-2-50-ver(…): reconstructing file:   0%|          |  0.00B / 26.0MB            

aa_dataset-tickets-multi-lang-5-2-50-ver(…): downloading bytes:           |  0.00B            

(…)set-tickets-german_normalized_50_5_2.csv:   0%|          | 0.00/8.33M [00:00<?, ?B/s]

dataset-tickets-multi-lang-4-20k.csv: reconstructing file:   0%|          |  0.00B / 18.8MB            

dataset-tickets-multi-lang-4-20k.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/61765 [00:00<?, ? examples/s]

Colonnes originales : ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']
Nombre total de lignes : 61765


Filter:   0%|          | 0/61765 [00:00<?, ? examples/s]

Nombre de tickets IT retenus : 27477
Catégories retenues : ['IT Support', 'Product Support', 'Service Outages and Maintenance', 'Technical Support']


## 4. Tokenizer BPE (from scratch, entraîné sur le corpus HF réel)

In [11]:
import json
import re
from collections import Counter, defaultdict

PAD, UNK, CLS, SEP = "<pad>", "<unk>", "<cls>", "<sep>"
SPECIAL_TOKENS = [PAD, UNK, CLS, SEP]

def _word_to_symbols(word):
    return list(word) + ["</w>"]

class BPETokenizer:
    def __init__(self, vocab_size=3000):
        self.vocab_size = vocab_size
        self.token2id = {}
        self.id2token = {}
        self.merges = []

    def train(self, texts):
        word_freq = Counter()
        for text in texts:
            words = re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)
            word_freq.update(words)

        vocab = {tuple(_word_to_symbols(w)): f for w, f in word_freq.items()}
        base_chars = set()
        for symbols in vocab:
            base_chars.update(symbols)

        num_merges = max(0, self.vocab_size - len(base_chars) - len(SPECIAL_TOKENS))

        for _ in range(num_merges):
            pairs = defaultdict(int)
            for symbols, freq in vocab.items():
                for i in range(len(symbols) - 1):
                    pairs[(symbols[i], symbols[i + 1])] += freq
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            if pairs[best_pair] < 2:
                break
            self.merges.append(best_pair)

            new_vocab = {}
            a, b = best_pair
            merged = a + b
            for symbols, freq in vocab.items():
                new_symbols = []
                i = 0
                while i < len(symbols):
                    if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                        new_symbols.append(merged)
                        i += 2
                    else:
                        new_symbols.append(symbols[i])
                        i += 1
                new_vocab[tuple(new_symbols)] = freq
            vocab = new_vocab

        tokens = list(SPECIAL_TOKENS) + sorted(base_chars)
        for a, b in self.merges:
            merged = a + b
            if merged not in tokens:
                tokens.append(merged)

        self.token2id = {t: i for i, t in enumerate(tokens)}
        self.id2token = {i: t for t, i in self.token2id.items()}

    def _bpe_word(self, word):
        symbols = _word_to_symbols(word)
        for a, b in self.merges:
            i = 0
            merged = a + b
            new_symbols = []
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                    new_symbols.append(merged)
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols
        return symbols

    def encode(self, text, max_len=96, add_special_tokens=True):
        words = re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)
        ids = []
        if add_special_tokens:
            ids.append(self.token2id[CLS])
        for w in words:
            for sym in self._bpe_word(w):
                ids.append(self.token2id.get(sym, self.token2id[UNK]))
        if add_special_tokens:
            ids.append(self.token2id[SEP])

        ids = ids[:max_len]
        attention_mask = [1] * len(ids)
        pad_id = self.token2id[PAD]
        while len(ids) < max_len:
            ids.append(pad_id)
            attention_mask.append(0)
        return ids, attention_mask

    @property
    def vocab_size_(self):
        return len(self.token2id)

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"token2id": self.token2id, "merges": self.merges}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=len(data["token2id"]))
        tok.token2id = data["token2id"]
        tok.id2token = {int(v): k for k, v in tok.token2id.items()}
        tok.merges = [tuple(m) for m in data["merges"]]
        return tok


## 5. Dataset PyTorch branché directement sur le `datasets.Dataset` HF (pas de CSV intermédiaire)

In [12]:
from torch.utils.data import Dataset, DataLoader, random_split

class HFTicketDataset(Dataset):
    """Pré-tokenise TOUT le corpus une seule fois à la création du Dataset,
    au lieu de retokeniser à chaque __getitem__ (donc à chaque epoch).
    C'est ce qui évite de payer le coût du tokenizer 15 fois (une par epoch)."""

    def __init__(self, hf_dataset, tokenizer, max_len=128):
        self.category_ids = []
        self.priority_ids = []
        self.input_ids = []
        self.attention_masks = []

        for row in hf_dataset:
            text = f"{row['subject'] or ''}. {row['body'] or ''}"
            input_ids, attention_mask = tokenizer.encode(text, max_len=max_len)
            self.input_ids.append(input_ids)
            self.attention_masks.append(attention_mask)
            self.category_ids.append(CATEGORY2ID[row["queue"]])
            self.priority_ids.append(PRIORITY2ID[normalize_priority(row["priority"])])

        # Conversion en tenseurs une seule fois (accès quasi instantané ensuite)
        self.input_ids = torch.tensor(self.input_ids, dtype=torch.long)
        self.attention_masks = torch.tensor(self.attention_masks, dtype=torch.long)
        self.category_ids = torch.tensor(self.category_ids, dtype=torch.long)
        self.priority_ids = torch.tensor(self.priority_ids, dtype=torch.long)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "category_id": self.category_ids[idx],
            "priority_id": self.priority_ids[idx],
        }

## 6. Architecture custom : ITSentinelNet (identique — self-attention codée à la main, multi-tâches)

In [13]:
import math
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :]
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        context = torch.matmul(attn, v)
        context = context.transpose(1, 2).contiguous().view(B, T, -1)
        return self.out_proj(context)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        x = x + self.dropout1(self.attn(self.norm1(x), key_padding_mask))
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x

class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.key_proj = nn.Linear(d_model, d_model)

    def forward(self, x, key_padding_mask=None):
        B = x.size(0)
        q = self.query.expand(B, -1, -1)
        k = self.key_proj(x)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(x.size(-1))
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, :]
            scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        return torch.matmul(weights, x).squeeze(1)

class ITSentinelNet(nn.Module):
    def __init__(self, vocab_size, max_len=96, d_model=192, n_heads=4, n_layers=4,
                 d_ff=384, n_categories=4, n_priorities=3, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerEncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.pool = AttentionPooling(d_model)
        self.category_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, n_categories)
        )
        self.priority_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model // 2, n_priorities)
        )
        self.max_len = max_len
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        for block in self.blocks:
            x = block(x, key_padding_mask=attention_mask)
        x = self.final_norm(x)
        pooled = self.pool(x, key_padding_mask=attention_mask)
        return self.category_head(pooled), self.priority_head(pooled)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


In [14]:
"""
tokenizer.py
------------
Tokenizer BPE (Byte Pair Encoding) implémenté entièrement from scratch.

Aucune dépendance externe (pas de `tokenizers`, pas de `transformers`) :
le vocabulaire est appris directement sur le corpus IT fourni par l'utilisateur.
Cela garantit qu'aucun poids ni vocabulaire pré-entraîné n'est réutilisé.
"""

import json
import re
from collections import Counter, defaultdict

PAD, UNK, CLS, SEP = "<pad>", "<unk>", "<cls>", "<sep>"
SPECIAL_TOKENS = [PAD, UNK, CLS, SEP]


def _word_to_symbols(word):
    return list(word) + ["</w>"]


class BPETokenizer:
    def __init__(self, vocab_size=2000):
        self.vocab_size = vocab_size
        self.token2id = {}
        self.id2token = {}
        self.merges = []
        self._merge_rank = {}  # {(a,b): rang} — reconstruit après train()/load()

    def train(self, texts):
        word_freq = Counter()
        for text in texts:
            words = re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)
            word_freq.update(words)

        vocab = {tuple(_word_to_symbols(w)): f for w, f in word_freq.items()}
        base_chars = set()
        for symbols in vocab:
            base_chars.update(symbols)

        num_merges = max(0, self.vocab_size - len(base_chars) - len(SPECIAL_TOKENS))

        for _ in range(num_merges):
            pairs = defaultdict(int)
            for symbols, freq in vocab.items():
                for i in range(len(symbols) - 1):
                    pairs[(symbols[i], symbols[i + 1])] += freq
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            if pairs[best_pair] < 2:
                break
            self.merges.append(best_pair)

            new_vocab = {}
            a, b = best_pair
            merged = a + b
            for symbols, freq in vocab.items():
                new_symbols = []
                i = 0
                while i < len(symbols):
                    if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                        new_symbols.append(merged)
                        i += 2
                    else:
                        new_symbols.append(symbols[i])
                        i += 1
                new_vocab[tuple(new_symbols)] = freq
            vocab = new_vocab

        tokens = list(SPECIAL_TOKENS) + sorted(base_chars)
        for a, b in self.merges:
            merged = a + b
            if merged not in tokens:
                tokens.append(merged)

        self.token2id = {t: i for i, t in enumerate(tokens)}
        self.id2token = {i: t for t, i in self.token2id.items()}
        self._merge_rank = {pair: rank for rank, pair in enumerate(self.merges)}

    def _bpe_word(self, word):
        # Corrigé : on s'arrête dès qu'aucune fusion ne s'applique plus, au lieu
        # de reparcourir les ~vocab_size fusions pour chaque mot à chaque appel.
        symbols = _word_to_symbols(word)
        merge_rank = self._merge_rank

        while len(symbols) > 1:
            best_pair, best_rank = None, None
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                rank = merge_rank.get(pair)
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_pair, best_rank = pair, rank
            if best_pair is None:
                break

            a, b = best_pair
            merged = a + b
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                    new_symbols.append(merged)
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols
        return symbols

    def encode(self, text, max_len=64, add_special_tokens=True):
        words = re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)
        ids = []
        if add_special_tokens:
            ids.append(self.token2id[CLS])
        for w in words:
            for sym in self._bpe_word(w):
                ids.append(self.token2id.get(sym, self.token2id[UNK]))
        if add_special_tokens:
            ids.append(self.token2id[SEP])

        ids = ids[:max_len]
        attention_mask = [1] * len(ids)
        pad_id = self.token2id[PAD]
        while len(ids) < max_len:
            ids.append(pad_id)
            attention_mask.append(0)
        return ids, attention_mask

    def decode(self, ids):
        tokens = [self.id2token.get(i, UNK) for i in ids if i != self.token2id.get(PAD)]
        text = "".join(tokens).replace("</w>", " ")
        for t in (CLS, SEP):
            text = text.replace(t, "")
        return text.strip()

    @property
    def vocab_size_(self):
        return len(self.token2id)

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"token2id": self.token2id, "merges": self.merges}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=len(data["token2id"]))
        tok.token2id = data["token2id"]
        tok.id2token = {int(v): k for k, v in tok.token2id.items()}
        tok.merges = [tuple(m) for m in data["merges"]]
        tok._merge_rank = {pair: rank for rank, pair in enumerate(tok.merges)}
        return tok

## 7. Entraînement sur le dataset HF réel

In [15]:
import os
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter


def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# ------------------------------------------------------------------ #
# Hyperparamètres (corrigés : batch_size sûr pour un A100, LR adapté)
# ------------------------------------------------------------------ #
EPOCHS = 15
BATCH_SIZE = 768          # sûr sur A100 40 Go (~9-10 Go estimés avec backward)
LR = 4e-4                 # cohérent avec ce batch size (au lieu de 1e-3 fixe sans warmup)
VOCAB_SIZE = 400
MAX_LEN = 128

# Configuration ~28M paramètres
D_MODEL = 512
N_HEADS = 8
N_LAYERS = 8
D_FF = 2048

NUM_WORKERS = 4
CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Optimisations matérielles A100
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")  # active TF32 sur les Tensor Cores

timings = {}
t_start = time.time()

# ------------------------------------------------------------------ #
# 1. Tokenizer BPE (pur Python, CPU — c'est ici que passe le plus de temps)
# ------------------------------------------------------------------ #
t0 = time.time()
texts_for_tokenizer = [f"{r['subject']}. {r['body']}" for r in it_ds]
tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE)
tokenizer.train(texts_for_tokenizer)
tokenizer.save(os.path.join(CHECKPOINT_DIR, "tokenizer.json"))
timings["tokenizer"] = time.time() - t0
print(f"[{timings['tokenizer']:.1f}s] Tokenizer entraîné : {tokenizer.vocab_size_} tokens "
      f"sur {len(texts_for_tokenizer)} tickets")

# ------------------------------------------------------------------ #
# 2. Dataset / DataLoader
# ------------------------------------------------------------------ #
t0 = time.time()
full_dataset = HFTicketDataset(it_ds, tokenizer, max_len=MAX_LEN)
val_size = max(1, int(0.1 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)
timings["dataloader_setup"] = time.time() - t0
print(f"[{timings['dataloader_setup']:.1f}s] {len(train_ds)} exemples train / {len(val_ds)} val "
      f"({len(train_loader)} batches/epoch)")

# ------------------------------------------------------------------ #
# 3. Pondération de la loss "priorité" (corrige le biais observé : le modèle
#    prédisait "Low" avec 99% de confiance même sur des pannes critiques,
#    signe d'un déséquilibre de classes dans le dataset source)
# ------------------------------------------------------------------ #
RAW_PRIORITY_MAP = {"low": "Low", "medium": "Medium", "high": "Critical"}

def normalize_priority(value):
    return RAW_PRIORITY_MAP.get(str(value).strip().lower(), "Medium")

priority_counts = Counter(normalize_priority(p) for p in it_ds["priority"])
print("Distribution des priorités :", dict(priority_counts))
priority_weights = torch.tensor(
    [1.0 / priority_counts[p] for p in PRIORITIES], dtype=torch.float32
)
priority_weights = priority_weights / priority_weights.sum() * len(PRIORITIES)
priority_weights = priority_weights.to(device)
print("Poids appliqués par classe de priorité :", priority_weights.tolist())

# ------------------------------------------------------------------ #
# 4. Modèle
# ------------------------------------------------------------------ #
t0 = time.time()
model = ITSentinelNet(
    vocab_size=tokenizer.vocab_size_, max_len=MAX_LEN, d_model=D_MODEL,
    n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
    n_categories=len(CATEGORIES), n_priorities=len(PRIORITIES),
).to(device)

model = torch.compile(model)
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
n_params = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
print(f"Nombre de paramètres entraînables : {n_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.1,
)
criterion_cat = nn.CrossEntropyLoss()
criterion_prio = nn.CrossEntropyLoss(weight=priority_weights)
timings["model_setup"] = time.time() - t0
print(f"[{timings['model_setup']:.1f}s] Modèle prêt (compilation différée au 1er batch)")

# ------------------------------------------------------------------ #
# 5. Évaluation
# ------------------------------------------------------------------ #
def evaluate(loader):
    model.eval()
    correct_cat, correct_prio, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            category_id = batch["category_id"].to(device, non_blocking=True)
            priority_id = batch["priority_id"].to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                cat_logits, prio_logits = model(input_ids, attention_mask)

            correct_cat += (cat_logits.argmax(-1) == category_id).sum().item()
            correct_prio += (prio_logits.argmax(-1) == priority_id).sum().item()
            total += category_id.size(0)
    model.train()
    return correct_cat / total, correct_prio / total

# ------------------------------------------------------------------ #
# 6. Boucle d'entraînement
# ------------------------------------------------------------------ #
best_val_acc = 0.0
t_train_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        category_id = batch["category_id"].to(device, non_blocking=True)
        priority_id = batch["priority_id"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            cat_logits, prio_logits = model(input_ids, attention_mask)
            loss = 0.7 * criterion_cat(cat_logits, category_id) + 0.3 * criterion_prio(prio_logits, priority_id)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_acc_cat, val_acc_prio = evaluate(val_loader)
    epoch_time = time.time() - epoch_start
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} "
          f"| val_acc_category={val_acc_cat:.3f} | val_acc_priority={val_acc_prio:.3f} "
          f"| lr={current_lr:.2e} | {epoch_time:.1f}s/epoch")

    combined_acc = 0.7 * val_acc_cat + 0.3 * val_acc_prio
    if combined_acc > best_val_acc:
        best_val_acc = combined_acc
        torch.save({
            "model_state_dict": raw_model.state_dict(),
            "config": {
                "vocab_size": tokenizer.vocab_size_, "max_len": MAX_LEN, "d_model": D_MODEL,
                "n_heads": N_HEADS, "n_layers": N_LAYERS, "d_ff": D_FF,
                "n_categories": len(CATEGORIES), "n_priorities": len(PRIORITIES),
            },
        }, os.path.join(CHECKPOINT_DIR, "itsentinel_best.pt"))
        print(f"  -> nouveau meilleur modèle sauvegardé (score combiné {combined_acc:.3f})")

timings["training"] = time.time() - t_train_start
timings["total"] = time.time() - t_start

print("\n" + "=" * 60)
print(f"Entraînement terminé. Meilleur score combiné : {round(best_val_acc, 3)}")
print("=" * 60)
print("Temps par étape :")
for step, seconds in timings.items():
    print(f"  {step:20s} : {seconds:.1f}s")

[26.6s] Tokenizer entraîné : 400 tokens sur 27477 tickets
[24.7s] 24730 exemples train / 2747 val (32 batches/epoch)
Distribution des priorités : {'Critical': 13422, 'Medium': 10269, 'Low': 3786}
Poids appliqués par classe de priorité : [1.8173484802246094, 0.6700244545936584, 0.512627124786377]
Nombre de paramètres entraînables : 26,150,407
[8.2s] Modèle prêt (compilation différée au 1er batch)
Epoch 01/15 | loss=1.2290 | val_acc_category=0.465 | val_acc_priority=0.270 | lr=3.11e-04 | 65.1s/epoch
  -> nouveau meilleur modèle sauvegardé (score combiné 0.406)
Epoch 02/15 | loss=1.1554 | val_acc_category=0.478 | val_acc_priority=0.471 | lr=3.98e-04 | 5.3s/epoch
  -> nouveau meilleur modèle sauvegardé (score combiné 0.476)
Epoch 03/15 | loss=1.0904 | val_acc_category=0.520 | val_acc_priority=0.455 | lr=3.87e-04 | 5.3s/epoch
  -> nouveau meilleur modèle sauvegardé (score combiné 0.501)
Epoch 04/15 | loss=1.0554 | val_acc_category=0.523 | val_acc_priority=0.405 | lr=3.66e-04 | 5.3s/epoch
Ep

In [16]:
import time
import datetime

if 'epoch_start' in locals() and 'epoch' in locals() and 'EPOCHS' in locals():
    time_per_epoch = time.time() - epoch_start
    epochs_remaining = EPOCHS - epoch + 1
    estimated_remaining_seconds = time_per_epoch * epochs_remaining
    print(f"Temps écoulé pour l'époque courante : {time_per_epoch:.2f} secondes")
    print(f"Estimation du temps restant : {datetime.timedelta(seconds=int(estimated_remaining_seconds))}")
else:
    print("L'entraînement n'a pas encore commencé ou les variables ne sont pas disponibles.")

Temps écoulé pour l'époque courante : 5.31 secondes
Estimation du temps restant : 0:00:05


In [17]:
import gc
import torch

# Nettoyage des variables résiduelles de la boucle d'entraînement si elles existent
try:
    del cat_logits, prio_logits, loss, batch, input_ids, attention_mask
except NameError:
    pass

# Forcer le garbage collector de Python
gc.collect()

# Vider le cache de la VRAM de PyTorch
torch.cuda.empty_cache()

print(f"Mémoire VRAM actuellement allouée : {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"Mémoire VRAM en cache (réservée)  : {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
print("VRAM nettoyée avec succès !")

Mémoire VRAM actuellement allouée : 422.16 MB
Mémoire VRAM en cache (réservée)  : 556.00 MB
VRAM nettoyée avec succès !


## 8. Inférence

In [18]:
def load_best_model():
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "itsentinel_best.pt"), map_location=device)
    tok = BPETokenizer.load(os.path.join(CHECKPOINT_DIR, "tokenizer.json"))
    m = ITSentinelNet(**ckpt["config"]).to(device)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    return m, tok

def predict(text, m=None, tok=None, max_len=MAX_LEN):
    if m is None or tok is None:
        m, tok = load_best_model()
    input_ids, attention_mask = tok.encode(text, max_len=max_len)
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)
    attention_mask = torch.tensor([attention_mask], dtype=torch.long).to(device)
    with torch.no_grad():
        cat_logits, prio_logits = m(input_ids, attention_mask)
        cat_probs = F.softmax(cat_logits, dim=-1)[0]
        prio_probs = F.softmax(prio_logits, dim=-1)[0]
    return {
        "category": CATEGORIES[cat_probs.argmax().item()],
        "category_confidence": cat_probs.max().item(),
        "priority": PRIORITIES[prio_probs.argmax().item()],
        "priority_confidence": prio_probs.max().item(),
    }

best_model, best_tokenizer = load_best_model()

exemples = [
    "VPN connectivity is down for all remote employees since this morning.",
    "Der Server antwortet seit 20 Minuten nicht mehr, kritischer Ausfall.",
    "The firewall is blocking outbound HTTPS traffic for the whole office.",
]
for texte in exemples:
    result = predict(texte, best_model, best_tokenizer)
    print(f"Texte     : {texte}")
    print(f"Catégorie : {result['category']} ({result['category_confidence']:.1%})")
    print(f"Priorité  : {result['priority']} ({result['priority_confidence']:.1%})")
    print("-" * 60)


Texte     : VPN connectivity is down for all remote employees since this morning.
Catégorie : IT Support (96.4%)
Priorité  : Medium (93.4%)
------------------------------------------------------------
Texte     : Der Server antwortet seit 20 Minuten nicht mehr, kritischer Ausfall.
Catégorie : Service Outages and Maintenance (80.0%)
Priorité  : Critical (70.3%)
------------------------------------------------------------
Texte     : The firewall is blocking outbound HTTPS traffic for the whole office.
Catégorie : IT Support (77.3%)
Priorité  : Critical (56.9%)
------------------------------------------------------------


## 9. Étendre avec les 9 autres datasets

Chaque dataset a un schéma différent — utilisez `inspect_dataset(key)` (section 2) pour
découvrir ses colonnes, puis écrivez un petit adaptateur similaire à `HFTicketDataset`.
Quelques pistes concrètes :

- **`network_anomaly_fr`**, **`network_anomaly`** : ajoutez une tâche de classification
  "type d'anomalie réseau" en plus de catégorie/priorité (modèle 3 têtes).
- **`hdfs_logs`** : la colonne `label` (Normal/Anomaly) peut remplacer ou compléter la tête
  `priority_head` pour une détection d'anomalie binaire sur des séquences de logs.
- **`cybersec_fenrir`** : format instruction (system/user/assistant) — utile pour enrichir
  le **vocabulaire du tokenizer** (plus de termes de cybersécurité) même sans réutiliser
  les labels directement.
- **`code_vuln_devign`** : remplace le texte "ticket" par du code source Python et le label
  catégorie par "vulnérable / non-vulnérable" — même architecture, nouvelle tâche.
- **Combiner plusieurs datasets** : concaténez les textes de 2-3 datasets avant d'entraîner
  le tokenizer BPE (`tokenizer.train(texts_a + texts_b + texts_c)`) pour un vocabulaire plus
  riche, puis entraînez un modèle par tâche ou un modèle multi-tâches à N têtes.


## 10. (Optionnel) Sauvegarder sur Google Drive

In [19]:
from google.colab import drive
drive.mount("/content/drive")
import shutil
shutil.copytree("/content/checkpoints", "/content/drive/MyDrive/ITSentinelNet_hf_checkpoints", dirs_exist_ok=True)
print("Checkpoints copiés sur Google Drive avec succès !")

ValueError: mount failed

In [ ]:
# ============================================================================
# ITSentinelNet — script complet unique (Google Colab, GPU A100)
#
# Réseau de neurones CUSTOM en PyTorch (tokenizer BPE from scratch + Transformer
# encodeur codé à la main, AUCUN fine-tuning, AUCUN poids pré-entraîné).
# Entraîné sur le vrai dataset Hugging Face "Tobi-Bueck/customer-support-tickets"
# (tickets support IT/technique réels, EN/DE), en multi-tâches :
#     - category  : file du ticket (Technical Support, IT Support, ...)
#     - priority  : Low / Medium / Critical
#
# Corrections appliquées suite aux tests sur A100 :
#   - tokenizer BPE optimisé (arrêt anticipé par rang de fusion, ~100x plus
#     rapide qu'une implémentation naïve qui reboucle sur tout le vocabulaire)
#   - pré-tokenisation UNE SEULE FOIS avant l'entraînement (pas à chaque epoch)
#   - batch_size=384 (sûr sur A100 40 Go, au lieu d'un batch trop gros -> OOM)
#   - LR + OneCycleLR (warmup) adaptés à ce batch size
#   - pondération de la loss "priority" par classe (dataset déséquilibré)
#   - bf16 autocast + TF32 + torch.compile (optimisations natives A100)
# ============================================================================

# --- Installation ---
!pip install -q datasets

import os
import re
import json
import math
import time
import random
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from datasets import load_dataset

# ============================================================================
# 0. Setup GPU
# ============================================================================
print("PyTorch version:", torch.__version__)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun GPU détecté")
assert torch.cuda.is_available(), "Active un GPU (A100) via Exécution > Modifier le type d'exécution."
device = torch.device("cuda")

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")  # active TF32 sur les Tensor Cores

def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# ============================================================================
# 1. Chargement du dataset IT réel (Hugging Face Hub)
# ============================================================================
t0 = time.time()
USE_ALL_QUEUES = True  # True = garde les 10 files (IT + non-IT), False = IT uniquement

raw_ds = load_dataset("Tobi-Bueck/customer-support-tickets", split="train")
print("Colonnes originales :", raw_ds.column_names)
print("Nombre total de lignes :", len(raw_ds))

IT_QUEUES = {"Technical Support", "Product Support", "IT Support", "Service Outages and Maintenance"}

def is_valid(example):
    if example["subject"] is None or example["body"] is None or example["priority"] is None:
        return False
    if USE_ALL_QUEUES:
        return True
    return example["queue"] in IT_QUEUES

it_ds = raw_ds.filter(is_valid, num_proc=4)
print(f"[{time.time()-t0:.1f}s] Nombre de tickets retenus : {len(it_ds)}")

CATEGORIES = sorted(set(it_ds["queue"]))
PRIORITIES = ["Low", "Medium", "Critical"]
CATEGORY2ID = {c: i for i, c in enumerate(CATEGORIES)}
PRIORITY2ID = {p: i for i, p in enumerate(PRIORITIES)}

RAW_PRIORITY_MAP = {"low": "Low", "medium": "Medium", "high": "Critical"}

def normalize_priority(value):
    return RAW_PRIORITY_MAP.get(str(value).strip().lower(), "Medium")

print(f"Catégories ({len(CATEGORIES)}) : {CATEGORIES}")
priority_counts = Counter(normalize_priority(p) for p in it_ds["priority"])
print("Distribution des priorités :", dict(priority_counts))

# ============================================================================
# 2. Tokenizer BPE — from scratch, optimisé (arrêt anticipé par rang)
# ============================================================================
PAD, UNK, CLS, SEP = "<pad>", "<unk>", "<cls>", "<sep>"
SPECIAL_TOKENS = [PAD, UNK, CLS, SEP]


def _word_to_symbols(word):
    return list(word) + ["</w>"]


class BPETokenizer:
    def __init__(self, vocab_size=4000):
        self.vocab_size = vocab_size
        self.token2id = {}
        self.id2token = {}
        self.merges = []
        self._merge_rank = {}

    def train(self, texts):
        word_freq = Counter()
        for text in texts:
            words = re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)
            word_freq.update(words)

        vocab = {tuple(_word_to_symbols(w)): f for w, f in word_freq.items()}
        base_chars = set()
        for symbols in vocab:
            base_chars.update(symbols)

        num_merges = max(0, self.vocab_size - len(base_chars) - len(SPECIAL_TOKENS))

        for _ in range(num_merges):
            pairs = defaultdict(int)
            for symbols, freq in vocab.items():
                for i in range(len(symbols) - 1):
                    pairs[(symbols[i], symbols[i + 1])] += freq
            if not pairs:
                break
            best_pair = max(pairs, key=pairs.get)
            if pairs[best_pair] < 2:
                break
            self.merges.append(best_pair)

            new_vocab = {}
            a, b = best_pair
            merged = a + b
            for symbols, freq in vocab.items():
                new_symbols = []
                i = 0
                while i < len(symbols):
                    if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                        new_symbols.append(merged)
                        i += 2
                    else:
                        new_symbols.append(symbols[i])
                        i += 1
                new_vocab[tuple(new_symbols)] = freq
            vocab = new_vocab

        tokens = list(SPECIAL_TOKENS) + sorted(base_chars)
        for a, b in self.merges:
            merged = a + b
            if merged not in tokens:
                tokens.append(merged)

        self.token2id = {t: i for i, t in enumerate(tokens)}
        self.id2token = {i: t for t, i in self.token2id.items()}
        self._merge_rank = {pair: rank for rank, pair in enumerate(self.merges)}

    def _bpe_word(self, word):
        symbols = _word_to_symbols(word)
        merge_rank = self._merge_rank

        while len(symbols) > 1:
            best_pair, best_rank = None, None
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                rank = merge_rank.get(pair)
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_pair, best_rank = pair, rank
            if best_pair is None:
                break

            a, b = best_pair
            merged = a + b
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                    new_symbols.append(merged)
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols
        return symbols

    def encode(self, text, max_len=128, add_special_tokens=True):
        words = re.findall(r"\w+|[^\w\s]", text.lower(), re.UNICODE)
        ids = []
        if add_special_tokens:
            ids.append(self.token2id[CLS])
        for w in words:
            for sym in self._bpe_word(w):
                ids.append(self.token2id.get(sym, self.token2id[UNK]))
        if add_special_tokens:
            ids.append(self.token2id[SEP])

        ids = ids[:max_len]
        attention_mask = [1] * len(ids)
        pad_id = self.token2id[PAD]
        while len(ids) < max_len:
            ids.append(pad_id)
            attention_mask.append(0)
        return ids, attention_mask

    def decode(self, ids):
        tokens = [self.id2token.get(i, UNK) for i in ids if i != self.token2id.get(PAD)]
        text = "".join(tokens).replace("</w>", " ")
        for t in (CLS, SEP):
            text = text.replace(t, "")
        return text.strip()

    @property
    def vocab_size_(self):
        return len(self.token2id)

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"token2id": self.token2id, "merges": self.merges}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=len(data["token2id"]))
        tok.token2id = data["token2id"]
        tok.id2token = {int(v): k for k, v in tok.token2id.items()}
        tok.merges = [tuple(m) for m in data["merges"]]
        tok._merge_rank = {pair: rank for rank, pair in enumerate(tok.merges)}
        return tok


# ============================================================================
# 3. Dataset PyTorch — pré-tokenisé une seule fois (pas à chaque epoch)
# ============================================================================
class HFTicketDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len=128):
        self.category_ids = []
        self.priority_ids = []
        input_ids_list = []
        attention_masks_list = []

        for row in hf_dataset:
            text = f"{row['subject'] or ''}. {row['body'] or ''}"
            input_ids, attention_mask = tokenizer.encode(text, max_len=max_len)
            input_ids_list.append(input_ids)
            attention_masks_list.append(attention_mask)
            self.category_ids.append(CATEGORY2ID[row["queue"]])
            self.priority_ids.append(PRIORITY2ID[normalize_priority(row["priority"])])

        self.input_ids = torch.tensor(input_ids_list, dtype=torch.long)
        self.attention_masks = torch.tensor(attention_masks_list, dtype=torch.long)
        self.category_ids = torch.tensor(self.category_ids, dtype=torch.long)
        self.priority_ids = torch.tensor(self.priority_ids, dtype=torch.long)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "category_id": self.category_ids[idx],
            "priority_id": self.priority_ids[idx],
        }


# ============================================================================
# 4. Architecture custom : ITSentinelNet
#    (self-attention codée à la main, blocs pre-norm résiduels, pooling par
#    attention apprise, deux têtes multi-tâches)
# ============================================================================
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :]
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        context = torch.matmul(attn, v)
        context = context.transpose(1, 2).contiguous().view(B, T, -1)
        return self.out_proj(context)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        x = x + self.dropout1(self.attn(self.norm1(x), key_padding_mask))
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.key_proj = nn.Linear(d_model, d_model)

    def forward(self, x, key_padding_mask=None):
        B = x.size(0)
        q = self.query.expand(B, -1, -1)
        k = self.key_proj(x)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(x.size(-1))
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, :]
            scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        return torch.matmul(weights, x).squeeze(1)


class ITSentinelNet(nn.Module):
    def __init__(self, vocab_size, max_len=128, d_model=512, n_heads=8, n_layers=8,
                 d_ff=2048, n_categories=4, n_priorities=3, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerEncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.pool = AttentionPooling(d_model)
        self.category_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, n_categories)
        )
        self.priority_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model // 2, n_priorities)
        )
        self.max_len = max_len
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        for block in self.blocks:
            x = block(x, key_padding_mask=attention_mask)
        x = self.final_norm(x)
        pooled = self.pool(x, key_padding_mask=attention_mask)
        return self.category_head(pooled), self.priority_head(pooled)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================================================
# 5. Entraînement
# ============================================================================
EPOCHS = 15
BATCH_SIZE = 384          # sûr sur A100 40 Go (~9-10 Go estimés avec backward)
LR = 4e-4                 # cohérent avec ce batch size (OneCycleLR gère le warmup)
VOCAB_SIZE = 18000
MAX_LEN = 128

D_MODEL = 512
N_HEADS = 8
N_LAYERS = 8
D_FF = 2048

NUM_WORKERS = 4
CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

timings = {}
t_start = time.time()

# --- Tokenizer ---
t0 = time.time()
texts_for_tokenizer = [f"{r['subject']}. {r['body']}" for r in it_ds]
tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE)
tokenizer.train(texts_for_tokenizer)
tokenizer.save(os.path.join(CHECKPOINT_DIR, "tokenizer.json"))
timings["tokenizer_train"] = time.time() - t0
print(f"[{timings['tokenizer_train']:.1f}s] Tokenizer entraîné : {tokenizer.vocab_size_} tokens "
      f"sur {len(texts_for_tokenizer)} tickets")

# --- Pré-tokenisation + Dataset / DataLoader ---
t0 = time.time()
full_dataset = HFTicketDataset(it_ds, tokenizer, max_len=MAX_LEN)
val_size = max(1, int(0.1 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)
timings["dataset_setup"] = time.time() - t0
print(f"[{timings['dataset_setup']:.1f}s] {len(train_ds)} exemples train / {len(val_ds)} val "
      f"({len(train_loader)} batches/epoch)")

# --- Pondération de la loss "priorité" (dataset déséquilibré) ---
priority_weights = torch.tensor(
    [1.0 / max(priority_counts.get(p, 0), 1) for p in PRIORITIES], dtype=torch.float32
)
priority_weights = (priority_weights / priority_weights.sum() * len(PRIORITIES)).to(device)
print("Poids appliqués par classe de priorité :", priority_weights.tolist())

# --- Modèle ---
t0 = time.time()
model = ITSentinelNet(
    vocab_size=tokenizer.vocab_size_, max_len=MAX_LEN, d_model=D_MODEL,
    n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
    n_categories=len(CATEGORIES), n_priorities=len(PRIORITIES),
).to(device)

model = torch.compile(model)
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
n_params = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)
print(f"Nombre de paramètres entraînables : {n_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.1,
)
criterion_cat = nn.CrossEntropyLoss()
criterion_prio = nn.CrossEntropyLoss(weight=priority_weights)
timings["model_setup"] = time.time() - t0
print(f"[{timings['model_setup']:.1f}s] Modèle prêt (compilation différée au 1er batch)")


def evaluate(loader):
    model.eval()
    correct_cat, correct_prio, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            category_id = batch["category_id"].to(device, non_blocking=True)
            priority_id = batch["priority_id"].to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                cat_logits, prio_logits = model(input_ids, attention_mask)

            correct_cat += (cat_logits.argmax(-1) == category_id).sum().item()
            correct_prio += (prio_logits.argmax(-1) == priority_id).sum().item()
            total += category_id.size(0)
    model.train()
    return correct_cat / total, correct_prio / total


best_val_acc = 0.0
t_train_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        category_id = batch["category_id"].to(device, non_blocking=True)
        priority_id = batch["priority_id"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            cat_logits, prio_logits = model(input_ids, attention_mask)
            loss = 0.7 * criterion_cat(cat_logits, category_id) + 0.3 * criterion_prio(prio_logits, priority_id)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_acc_cat, val_acc_prio = evaluate(val_loader)
    epoch_time = time.time() - epoch_start
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} "
          f"| val_acc_category={val_acc_cat:.3f} | val_acc_priority={val_acc_prio:.3f} "
          f"| lr={current_lr:.2e} | {epoch_time:.1f}s/epoch")

    combined_acc = 0.7 * val_acc_cat + 0.3 * val_acc_prio
    if combined_acc > best_val_acc:
        best_val_acc = combined_acc
        torch.save({
            "model_state_dict": raw_model.state_dict(),
            "config": {
                "vocab_size": tokenizer.vocab_size_, "max_len": MAX_LEN, "d_model": D_MODEL,
                "n_heads": N_HEADS, "n_layers": N_LAYERS, "d_ff": D_FF,
                "n_categories": len(CATEGORIES), "n_priorities": len(PRIORITIES),
            },
        }, os.path.join(CHECKPOINT_DIR, "itsentinel_best.pt"))
        print(f"  -> nouveau meilleur modèle sauvegardé (score combiné {combined_acc:.3f})")

timings["training"] = time.time() - t_train_start
timings["total"] = time.time() - t_start

print("\n" + "=" * 60)
print(f"Entraînement terminé. Meilleur score combiné : {round(best_val_acc, 3)}")
print("=" * 60)
print("Temps par étape :")
for step, seconds in timings.items():
    print(f"  {step:20s} : {seconds:.1f}s")

# ============================================================================
# 6. Inférence
# ============================================================================
def load_best_model():
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "itsentinel_best.pt"), map_location=device)
    tok = BPETokenizer.load(os.path.join(CHECKPOINT_DIR, "tokenizer.json"))
    m = ITSentinelNet(**ckpt["config"]).to(device)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    return m, tok


def predict(text, m=None, tok=None, max_len=MAX_LEN):
    if m is None or tok is None:
        m, tok = load_best_model()
    input_ids, attention_mask = tok.encode(text, max_len=max_len)
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)
    attention_mask = torch.tensor([attention_mask], dtype=torch.long).to(device)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        cat_logits, prio_logits = m(input_ids, attention_mask)
        cat_probs = F.softmax(cat_logits.float(), dim=-1)[0]
        prio_probs = F.softmax(prio_logits.float(), dim=-1)[0]
    return {
        "category": CATEGORIES[cat_probs.argmax().item()],
        "category_confidence": cat_probs.max().item(),
        "priority": PRIORITIES[prio_probs.argmax().item()],
        "priority_confidence": prio_probs.max().item(),
    }


best_model, best_tokenizer = load_best_model()

exemples = [
    "VPN connectivity is down for all remote employees since this morning.",
    "Der Server antwortet seit 20 Minuten nicht mehr, kritischer Ausfall.",
    "The firewall is blocking outbound HTTPS traffic for the whole office.",
]
for texte in exemples:
    result = predict(texte, best_model, best_tokenizer)
    print(f"Texte     : {texte}")
    print(f"Catégorie : {result['category']} ({result['category_confidence']:.1%})")
    print(f"Priorité  : {result['priority']} ({result['priority_confidence']:.1%})")
    print("-" * 60)

# ============================================================================
# 7. (Optionnel) Sauvegarder sur Google Drive
# ============================================================================
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copytree(CHECKPOINT_DIR, "/content/drive/MyDrive/ITSentinelNet_checkpoints", dirs_exist_ok=True)
# print("Checkpoints copiés sur Google Drive.")

In [ ]:
from collections import Counter, defaultdict
import torch

# 1. Y a-t-il un confondu langue / priorité dans les données d'entraînement ?
lang_priority = defaultdict(Counter)
for row in it_ds:
    lang_priority[row["language"]][normalize_priority(row["priority"])] += 1

print("Distribution priorité par langue :")
for lang, counts in lang_priority.items():
    total = sum(counts.values())
    print(f"  {lang}: " + ", ".join(f"{p}={c} ({c/total:.0%})" for p, c in counts.items()))

# 2. Matrice de confusion sur la validation (catégorie ET priorité)
model.eval()
cat_confusion = torch.zeros(len(CATEGORIES), len(CATEGORIES), dtype=torch.long)
prio_confusion = torch.zeros(len(PRIORITIES), len(PRIORITIES), dtype=torch.long)

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            cat_logits, prio_logits = model(input_ids, attention_mask)
        cat_pred = cat_logits.argmax(-1).cpu()
        prio_pred = prio_logits.argmax(-1).cpu()
        for true_c, pred_c in zip(batch["category_id"], cat_pred):
            cat_confusion[true_c, pred_c] += 1
        for true_p, pred_p in zip(batch["priority_id"], prio_pred):
            prio_confusion[true_p, pred_p] += 1
model.train()

print("\nMatrice de confusion PRIORITÉ (lignes=vrai, colonnes=prédit) :")
print("        " + "  ".join(f"{p:>9s}" for p in PRIORITIES))
for i, p in enumerate(PRIORITIES):
    print(f"{p:>8s} " + "  ".join(f"{prio_confusion[i,j].item():>9d}" for j in range(len(PRIORITIES))))

print("\nMatrice de confusion CATÉGORIE (lignes=vrai, colonnes=prédit) :")
print("        " + "  ".join(f"{c[:9]:>9s}" for c in CATEGORIES))
for i, c in enumerate(CATEGORIES):
    print(f"{c[:8]:>8s} " + "  ".join(f"{cat_confusion[i,j].item():>9d}" for j in range(len(CATEGORIES))))

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# S'assurer que le modèle est en mode évaluation
model.eval()

all_true_cat, all_pred_cat = [], []
all_true_prio, all_pred_prio = [], []

print("Génération des prédictions sur le set de validation...")
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            cat_logits, prio_logits = model(input_ids, attention_mask)

        all_pred_cat.extend(cat_logits.argmax(-1).cpu().numpy())
        all_true_cat.extend(batch["category_id"].numpy())

        all_pred_prio.extend(prio_logits.argmax(-1).cpu().numpy())
        all_true_prio.extend(batch["priority_id"].numpy())

print("\n" + "="*50)
print("RAPPORT DE CLASSIFICATION : CATÉGORIES")
print("="*50)
print(classification_report(
    all_true_cat,
    all_pred_cat,
    target_names=CATEGORIES,
    zero_division=0
))

print("\n" + "="*50)
print("RAPPORT DE CLASSIFICATION : PRIORITÉS")
print("="*50)
print(classification_report(
    all_true_prio,
    all_pred_prio,
    target_names=PRIORITIES,
    zero_division=0
))

## 11. Entraînement d'un modèle plus grand (~40M de paramètres)

Ici, on réutilise le tokenizer et les dataloaders déjà calculés en mémoire pour gagner du temps. On augmente simplement `N_LAYERS` à 10 pour atteindre environ ~41 millions de paramètres.

In [ ]:
import time
import torch
import os

print("--- Configuration du modèle ~40M paramètres ---")

# Hyperparamètres augmentés pour atteindre ~40M
D_MODEL_40M = 1024
N_HEADS_40M = 8
N_LAYERS_40M = 20  # 10 couches au lieu de 8 (ajoute ~6.3M de paramètres)
D_FF_40M = 2048
EPOCHS_40M = 15
LR_40M = 4e-4

# 1. Initialisation du nouveau modèle
model_40m = ITSentinelNet(
    vocab_size=tokenizer.vocab_size_,
    max_len=MAX_LEN,
    d_model=D_MODEL_40M,
    n_heads=N_HEADS_40M,
    n_layers=N_LAYERS_40M,
    d_ff=D_FF_40M,
    n_categories=len(CATEGORIES),
    n_priorities=len(PRIORITIES),
).to(device)

# Optimisation de la compilation
model_40m = torch.compile(model_40m)
raw_model_40m = model_40m._orig_mod if hasattr(model_40m, "_orig_mod") else model_40m

n_params_40m = sum(p.numel() for p in raw_model_40m.parameters() if p.requires_grad)
print(f"Nombre exact de paramètres du nouveau modèle : {n_params_40m:,}")

# 2. Optimiseur et Scheduler
optimizer_40m = torch.optim.AdamW(model_40m.parameters(), lr=LR_40M, weight_decay=0.01)
scheduler_40m = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_40m, max_lr=LR_40M, epochs=EPOCHS_40M, steps_per_epoch=len(train_loader), pct_start=0.1,
)

# 3. Boucle d'entraînement
best_val_acc_40m = 0.0
t_train_start = time.time()

print("\nDébut de l'entraînement...")
for epoch in range(1, EPOCHS_40M + 1):
    epoch_start = time.time()
    total_loss = 0.0
    model_40m.train()

    # Phase de Train
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        category_id = batch["category_id"].to(device, non_blocking=True)
        priority_id = batch["priority_id"].to(device, non_blocking=True)

        optimizer_40m.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            cat_logits, prio_logits = model_40m(input_ids, attention_mask)
            loss = 0.7 * criterion_cat(cat_logits, category_id) + 0.3 * criterion_prio(prio_logits, priority_id)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_40m.parameters(), max_norm=1.0)
        optimizer_40m.step()
        scheduler_40m.step()
        total_loss += loss.item()

    # Phase d'Evaluation
    model_40m.eval()
    correct_cat, correct_prio, total = 0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            category_id = batch["category_id"].to(device, non_blocking=True)
            priority_id = batch["priority_id"].to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                cat_logits, prio_logits = model_40m(input_ids, attention_mask)

            correct_cat += (cat_logits.argmax(-1) == category_id).sum().item()
            correct_prio += (prio_logits.argmax(-1) == priority_id).sum().item()
            total += category_id.size(0)

    val_acc_cat = correct_cat / total
    val_acc_prio = correct_prio / total
    epoch_time = time.time() - epoch_start
    current_lr = scheduler_40m.get_last_lr()[0]

    print(f"Epoch {epoch:02d}/{EPOCHS_40M} | loss={total_loss/len(train_loader):.4f} "
          f"| val_acc_category={val_acc_cat:.3f} | val_acc_priority={val_acc_prio:.3f} "
          f"| lr={current_lr:.2e} | {epoch_time:.1f}s/epoch")

    combined_acc = 0.7 * val_acc_cat + 0.3 * val_acc_prio

    # Sauvegarde du meilleur modèle
    if combined_acc > best_val_acc_40m:
        best_val_acc_40m = combined_acc
        torch.save({
            "model_state_dict": raw_model_40m.state_dict(),
            "config": {
                "vocab_size": tokenizer.vocab_size_, "max_len": MAX_LEN, "d_model": D_MODEL_40M,
                "n_heads": N_HEADS_40M, "n_layers": N_LAYERS_40M, "d_ff": D_FF_40M,
                "n_categories": len(CATEGORIES), "n_priorities": len(PRIORITIES),
            },
        }, os.path.join(CHECKPOINT_DIR, "itsentinel_40M_best.pt"))
        print(f"  -> nouveau meilleur modèle 40M sauvegardé (score combiné {combined_acc:.3f})")

print("\n" + "=" * 60)
print(f"Entraînement 40M terminé en {time.time() - t_train_start:.1f}s. Meilleur score : {best_val_acc_40m:.3f}")
print("=" * 60)

## 12. Test / Inférence avec le modèle 40M

Modifiez la liste `phrases_de_test` ci-dessous avec vos propres exemples pour tester le nouveau modèle.

In [ ]:
import torch
import torch.nn.functional as F
import os

MAX_LEN = 128

def load_best_model_40m():
    # Charger le checkpoint spécifique au 40M
    ckpt_path = os.path.join(CHECKPOINT_DIR, "itsentinel_40M_best.pt")
    ckpt = torch.load(ckpt_path, map_location=device)

    # Charger le tokenizer
    tok = BPETokenizer.load(os.path.join(CHECKPOINT_DIR, "tokenizer.json"))

    # Initialiser le modèle avec la configuration sauvegardée (qui contient n_layers=10, etc.)
    m = ITSentinelNet(**ckpt["config"]).to(device)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    return m, tok

def predict_40m(text, m, tok, max_len=MAX_LEN):
    input_ids, attention_mask = tok.encode(text, max_len=max_len)
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)
    attention_mask = torch.tensor([attention_mask], dtype=torch.long).to(device)

    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        cat_logits, prio_logits = m(input_ids, attention_mask)
        cat_probs = F.softmax(cat_logits.float(), dim=-1)[0]
        prio_probs = F.softmax(prio_logits.float(), dim=-1)[0]

    return {
        "category": CATEGORIES[cat_probs.argmax().item()],
        "category_confidence": cat_probs.max().item(),
        "priority": PRIORITIES[prio_probs.argmax().item()],
        "priority_confidence": prio_probs.max().item(),
    }

# --- Exécution du test ---
model_40m_eval, tok_eval = load_best_model_40m()

phrases_de_test = [
    "My laptop battery is swelling and the trackpad is popping out!",
    "I cannot access my emails since the password reset this morning.",
    "Le serveur de base de données principal a crashé, plus rien ne fonctionne, impact client direct !",
    "Need help installing the new version of Adobe Acrobat on my PC."
]

print("--- Prédictions du modèle 40M ---")
for texte in phrases_de_test:
    result = predict_40m(texte, model_40m_eval, tok_eval)
    print(f"Texte     : {texte}")
    print(f"Catégorie : {result['category']} ({result['category_confidence']:.1%})")
    print(f"Priorité  : {result['priority']} ({result['priority_confidence']:.1%})")
    print("-" * 60)